# Maps, Structs & Complex Types

**Dataset:** `samples.bakehouse.sales_transactions`, `samples.bakehouse.sales_franchises`, `samples.wanderbricks.clickstream`

**Difficulty:** Medium

**Topics:** `create_map`, `map_from_entries`, `map_keys`, `map_values`, `F.struct`, nested field access, `get_json_object`, `to_json`, `explode` on maps, struct columns

> **Why this matters:** Real-world data often comes as JSON, nested records, or key-value pairs.
> Mastering MapType and StructType lets you work with APIs, event logs, and semi-structured sources natively in Spark.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions  = spark.table("samples.bakehouse.sales_transactions")
df_franchises    = spark.table("samples.bakehouse.sales_franchises")
df_clickstream   = spark.table("samples.wanderbricks.clickstream")
df_customers     = spark.table("samples.bakehouse.sales_customers")

## Problem 1 — Create a Map Column

Using `F.create_map()`, build a `transaction_metadata` map column that stores two key-value pairs per row:
- `'product'` → the product name
- `'payment'` → the payment method

Then extract the product value back out using map subscript notation: `F.col('transaction_metadata')['product']`.

**Expected output columns:** `transactionID`, `transaction_metadata`, `product_from_map`

In [ ]:
# Problem 1 — write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'transaction_metadata' in cols, "Missing column: transaction_metadata"
assert 'product_from_map' in cols, "Missing column: product_from_map"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
# map type check
map_field = [f for f in result_1.schema.fields if f.name == 'transaction_metadata'][0]
assert isinstance(map_field.dataType, T.MapType), "transaction_metadata must be MapType"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — `map_from_entries`: Revenue Map per Franchise

Join `df_transactions` with `df_franchises` on `franchiseID`. Group by `franchiseID` and `name`.
For each franchise build a map of `{product → total_revenue}` using:
```python
F.map_from_entries(F.collect_list(F.struct(F.col('product'), F.col('revenue'))))
```
Also add `product_count` using `F.size()` on the resulting map.

**Expected output columns:** `franchiseID`, `name`, `product_revenue_map`, `product_count`

In [ ]:
# Problem 2 — write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'product_revenue_map' in cols, "Missing column: product_revenue_map"
assert 'product_count' in cols, "Missing column: product_count"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
map_field = [f for f in result_2.schema.fields if f.name == 'product_revenue_map'][0]
assert isinstance(map_field.dataType, T.MapType), "product_revenue_map must be MapType"
from pyspark.sql.functions import col
min_count = result_2.agg(F.min('product_count')).collect()[0][0]
assert min_count >= 1, "product_count should be >= 1"
print(f"Problem 2 passed ✓  ({cnt} franchises)")

## Problem 3 — Struct Column Creation

On `df_franchises`, create a `location` struct column containing `city`, `country`, and `zipcode` using `F.struct()`.
Then access the nested `city` field back out using `F.col('location.city')` to create `city_from_struct`.

**Expected output columns:** `franchiseID`, `name`, `location` (StructType), `city_from_struct`

In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'location' in cols, "Missing column: location"
assert 'city_from_struct' in cols, "Missing column: city_from_struct"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
loc_field = [f for f in result_3.schema.fields if f.name == 'location'][0]
assert isinstance(loc_field.dataType, T.StructType), "location must be StructType"
struct_fields = [f.name for f in loc_field.dataType.fields]
assert 'city' in struct_fields, "location struct must contain 'city'"
assert 'country' in struct_fields, "location struct must contain 'country'"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Access Struct Fields from Clickstream

The `samples.wanderbricks.clickstream` table has a `metadata` column of type `struct<device:string, referrer:string>`.
Extract both sub-fields using `F.col('metadata.device')` and `F.col('metadata.referrer')`.
Count events per `device` and `event` combination.

**Expected output columns:** `device`, `event`, `event_count`

In [ ]:
# Problem 4 — write your solution here
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'device' in cols, "Missing column: device"
assert 'event' in cols, "Missing column: event"
assert 'event_count' in cols, "Missing column: event_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
min_count = result_4.agg(F.min('event_count')).collect()[0][0]
assert min_count >= 1, "event_count must be >= 1"
print(f"Problem 4 passed ✓  ({cnt} device-event combinations)")

## Problem 5 — Payment Revenue Map per Franchise

For each `franchiseID`, build a map of `{paymentMethod → total_revenue}` from `df_transactions`.
Use `map_from_entries` + `collect_list`. Then use `F.map_keys()` to also show which payment methods each franchise accepts as an array.

**Expected output columns:** `franchiseID`, `revenue_by_payment` (MapType), `payment_methods` (ArrayType)

In [ ]:
# Problem 5 — write your solution here
# F.map_keys(col) returns an array of map keys
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'revenue_by_payment' in cols, "Missing column: revenue_by_payment"
assert 'payment_methods' in cols, "Missing column: payment_methods"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
map_field = [f for f in result_5.schema.fields if f.name == 'revenue_by_payment'][0]
assert isinstance(map_field.dataType, T.MapType), "revenue_by_payment must be MapType"
arr_field = [f for f in result_5.schema.fields if f.name == 'payment_methods'][0]
assert isinstance(arr_field.dataType, T.ArrayType), "payment_methods must be ArrayType"
print(f"Problem 5 passed ✓  ({cnt} franchises)")

## Problem 6 — Explode a Map

Take the revenue map from Problem 5 (or rebuild it). Use `F.explode(F.col('revenue_by_payment'))` to expand
the map back into one row per payment method per franchise. The explode of a MapType produces columns `key` and `value`.
Rename them to `payment_method` and `revenue`.

**Expected output columns:** `franchiseID`, `payment_method`, `revenue`

In [ ]:
# Problem 6 — write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'payment_method' in cols, "Missing column: payment_method"
assert 'revenue' in cols, "Missing column: revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# Exploded result should have more rows than the map DataFrame
assert cnt >= result_5.count(), "Exploded rows should be >= map rows"
print(f"Problem 6 passed ✓  ({cnt} franchise-payment rows)")

## Problem 7 — JSON String → Struct → Extract

Build a JSON string column from `df_customers` using `F.to_json(F.struct('first_name', 'country', 'gender'))`
to create a `json_payload` column.
Then use `F.get_json_object(F.col('json_payload'), '$.country')` to extract the country back out as `extracted_country`.
This simulates working with JSON-encoded data common in event streams and APIs.

**Expected output columns:** `customerID`, `json_payload`, `extracted_country`

In [ ]:
# Problem 7 — write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'json_payload' in cols, "Missing column: json_payload"
assert 'extracted_country' in cols, "Missing column: extracted_country"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
# json_payload should be a string type
json_field = [f for f in result_7.schema.fields if f.name == 'json_payload'][0]
assert isinstance(json_field.dataType, T.StringType), "json_payload must be StringType"
# extracted_country should not be all nulls
null_count = result_7.filter(F.col('extracted_country').isNull()).count()
assert null_count < cnt, "extracted_country is all nulls — check get_json_object path"
print(f"Problem 7 passed ✓  ({cnt} rows)")